Emotion corpus into ChunkedCorpus format.

In [1]:
%load_ext autoreload
%autoreload all

#import notebook helpers
from IPython.display import display,  clear_output
from tqdm import tqdm
import builtins

#import libraries
import sys, os.path
importPath = os.path.abspath('../../common')
if not importPath in sys.path:
    sys.path.append(importPath)

#define base paths
origBasePath = os.path.abspath("./original/ktu")
print(origBasePath)

preprocBasePath = os.path.abspath("./preproc/ktu")
print(preprocBasePath)

/home/user/Work/misijos/detector-all/detector-emotion/data/original/ktu
/home/user/Work/misijos/detector-all/detector-emotion/data/preproc/ktu


In [2]:
#define file paths
inFilePath = os.path.join(origBasePath, "corpus_emotions_2026-05-21.json")
print(inFilePath)

outFilePath = os.path.join(preprocBasePath, "corpus_emotions_2026-05-21-mixed.ranged.json")
print(outFilePath)

/home/user/Work/misijos/detector-all/detector-emotion/data/original/ktu/corpus_emotions_2026-05-21.json
/home/user/Work/misijos/detector-all/detector-emotion/data/preproc/ktu/corpus_emotions_2026-05-21-mixed.ranged.json


In [3]:
#read the input

#define schema of the input
from pydantic import BaseModel, Field
from typing import List
import json

class InText(BaseModel):
    comment_id : int
    content : str    
    emotionality : float

#read the input file
inTexts : List[InText] = []

with open(inFilePath, mode="r", encoding="utf-8") as file:
    inJson = json.load(file)
    inTexts = [InText(**it) for it in inJson]

In [4]:
#build the label set and quantization function

def quantLbl(emot: float) -> int:
	lbl = 0 if emot <= 0.5 else 1
	return lbl

#build label dictionary
labelLst = ['none', 'present']
labelDict = { idx: lbl for idx, lbl in enumerate(labelLst) }

#
display(labelDict)

#build label groups
numHeadLabels = [len(labelDict)]
display(numHeadLabels)

{0: 'none', 1: 'present'}

[2]

In [5]:
#build chunked corpus

from corpus.chunkedCorpus import ChunkedCorpus, ChunkedText, Chunk, ChunkSplitter

#label to assign to neutral chunks
labelNone = 0
print(f"Neutral label is: {labelNone}")

#
corpus = ChunkedCorpus(labelDefs = labelDict, labelGrps = numHeadLabels, texts = list())
for inTextIdx, inText in tqdm(enumerate(inTexts), "Input texts processed"):
    #skip empty texts, just in case
    if inText.content.strip() == 0:
        continue

    #build the labeled chunk covering all input text
    chunkLbl = quantLbl(inText.emotionality)
    chunks : List[Chunk] = [Chunk(start=0, len=len(inText.content), label=chunkLbl)]

    #build the chunked text and register with the corpus
    chunkedText = ChunkedText(id=inTextIdx, text=inText.content, chunks = chunks)
    corpus.texts.append(chunkedText)

Neutral label is: 0


Input texts processed: 2660it [00:00, 56576.89it/s]


In [6]:
#build mixed corpus

import numpy as np

#build loaded and neutral sets of samples
lblsNeutral = np.roll(np.array(corpus.labelGrps, np.int32).cumsum(), 1)
lblsNeutral[0] = 0
lblsNeutral = set(lblsNeutral)

loadedSampleIdxs = set()
neutralSampleIdxs = set()

for sampleIdx, sample in enumerate(corpus.texts):
	sampleLbls = set([x.label for x in sample.chunks])
	if sampleLbls == lblsNeutral:
		neutralSampleIdxs.add(sampleIdx)
	else:
		loadedSampleIdxs.add(sampleIdx)

#define mimum and maximum amount of samples to include in the mix
minSamplesInTheMix = 3
maxSamplesInTheMix = 4

#define desired number of mixed samples
maxNumMixedSamples = len(corpus.texts) * 2

#
mixSampCorpus = ChunkedCorpus(labelDefs=corpus.labelDefs, labelGrps=corpus.labelGrps)

#build mixed samples
unusedLoadedSampleIdxs = list(loadedSampleIdxs)
unusedNeutralSampleIdxs = list(neutralSampleIdxs)

from random import Random
rng = Random(0)

mixSampIdx = 0
prg = tqdm(desc="Building mixed samples", total=maxNumMixedSamples)

while mixSampIdx < maxNumMixedSamples:
	#generate amount of samples to include in the mix
	numSamplesInTheMix = rng.choice(range(minSamplesInTheMix, maxSamplesInTheMix+1))

	#snapshot current sample consumption state
	savedUnusedLoadedSampleIdxs = list(unusedLoadedSampleIdxs)
	savedUnusedNeutralSampleIdxs = list(unusedNeutralSampleIdxs)

	#choose chunks for the next mixed sample	
	mixSampChunkIdxs = []
	chunkIsLoaded = np.zeros((numSamplesInTheMix,), np.int32)
	for chunkIdx in range(numSamplesInTheMix):
		#decide what kind of sample to use next
		useLoadedSample = rng.choice([True, False])

		#choose next sample, add to result
		if useLoadedSample:
			sampleIdx = rng.choice(unusedLoadedSampleIdxs)
			unusedLoadedSampleIdxs.remove(sampleIdx)
			mixSampChunkIdxs.append(sampleIdx)

			#indicate we have used a loaded chunk
			chunkIsLoaded[chunkIdx] = 1 
		else:
			sampleIdx = rng.choice(unusedNeutralSampleIdxs)
			unusedNeutralSampleIdxs.remove(sampleIdx)
			mixSampChunkIdxs.append(sampleIdx)

		#restore sample source sets, if any became exhausted
		if len(unusedLoadedSampleIdxs) == 0:
			unusedLoadedSampleIdxs = list(loadedSampleIdxs)
		if len(unusedNeutralSampleIdxs) == 0:
			unusedNeutralSampleIdxs = list(neutralSampleIdxs)

	#did we produce a uniform sample? reject it
	isUniformSample = (chunkIsLoaded.sum() == 0) or (chunkIsLoaded.sum() == numSamplesInTheMix)
	
	if isUniformSample:
		#restore consumption state of base samples
		unusedLoadedSampleIdxs = savedUnusedLoadedSampleIdxs
		unusedNeutralSampleIdxs = savedUnusedNeutralSampleIdxs

	#sample is non-uniform, use it
	else:
		#build unified sample from the chunks
		offset = 0
		mixSampText = ""
		mixSampChunks = []

		for msChunkIdx in mixSampChunkIdxs:
			chunkSample = corpus.texts[msChunkIdx]

			#add space between last sample and new sample, if this is not first sample
			if mixSampText != "":
				mixSampText += " "
				offset +=1 

			#append text of current sample
			mixSampText += chunkSample.text

			#offset and take chunks from the current sample
			for chunk in chunkSample.chunks:
				offsetChunk = Chunk(start=chunk.start + offset, len=chunk.len, label=chunk.label)
				mixSampChunks.append(offsetChunk)

			#advance chunk offset for next round
			offset += len(chunkSample.text)

		#create new mixed sample, add to corpus
		mixSampChunkedText = ChunkedText(id=mixSampIdx, text=mixSampText, chunks=mixSampChunks)
		mixSampCorpus.texts.append(mixSampChunkedText)

		#advance to building next sample
		mixSampIdx += 1
		prg.update(1)

#save result
mixSampCorpus.saveToJson(outFilePath)

Building mixed samples:  85%|████████▍ | 4505/5320 [00:00<00:00, 45041.24it/s]